# 02 Regime Comparison: Quantile vs HMM

## Objective
Compare quantile and HMM regimes using already-computed alignment, model-selection, persistence, and confusion-table outputs.

## Input files
- `results/regimes/hmm/hmm_model_selection.parquet`
- `results/regimes/hmm/hmm_persistence_metrics.parquet`
- `results/regimes/hmm/hmm_quantile_comparison.parquet`
- `results/regimes/hmm/hmm_quantile_confusion_table.parquet`
- HMM and quantile label files

## Output folder
`reports/study_notebooks/figures/regime_comparison` and `reports/study_notebooks/tables`.

## Thesis relevance
This notebook explains why the regime methods are complementary rather than expected to match exactly.

## Analysis-only safety
This notebook never imports or calls STUMPY, STUMP/MSTUMP, HMM fitting, LoCoMotif search, or any other expensive experiment algorithm. It only reads saved result files and thesis-scope feature parquet files, then produces derived tables and figures.

**Quantile caveat.** The quantile regime outputs store `rolling_volatility_60` as the actual volatility column across all quantile method identifiers. Therefore, quantile regimes are interpreted as 60-period rolling-volatility regimes with different regime-count granularities rather than as separate 30/60/240 volatility-horizon experiments.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "HPC workflow" / "HPC_Regime_and_motif_discovery" / "notebooks" / "study"
if NOTEBOOK_DIR.exists() and str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from study_helpers import *

ensure_study_output_dirs()
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Project root:", PROJECT_ROOT)
print("Workflow root:", WORKFLOW_ROOT)
print("Study outputs:", REPORT_ROOT)


## Load HMM-vs-Quantile Comparison Tables


In [ ]:
quantile_dir = result_path("regimes", "quantile")
hmm_dir = result_path("regimes", "hmm")

paths = {
    "hmm_labels": resolve_existing_file(hmm_dir, "hmm_regime_labels.parquet"),
    "quantile_labels": resolve_existing_file(quantile_dir, "quantile_regime_labels.parquet"),
    "hmm_model_selection": resolve_existing_file(hmm_dir, "hmm_model_selection.parquet"),
    "hmm_persistence": resolve_existing_file(hmm_dir, "hmm_persistence_metrics.parquet"),
    "hmm_quantile_comparison": resolve_existing_file(hmm_dir, "hmm_quantile_comparison.parquet"),
    "hmm_quantile_confusion": resolve_existing_file(hmm_dir, "hmm_quantile_confusion_table.parquet"),
    "quantile_transitions": resolve_existing_file(quantile_dir, "quantile_transition_matrix.parquet"),
}
loaded = {name: safe_read_parquet(path) for name, path in paths.items()}
for name in ["hmm_labels", "quantile_labels"]:
    loaded[name] = coerce_timestamp(loaded[name])

inventory = pd.DataFrame([
    {"name": name, "path": str(path), "exists": path.exists(), "rows": len(loaded[name]), "columns": len(loaded[name].columns)}
    for name, path in paths.items()
])
display_table(inventory)
save_table(inventory, "study_regime_comparison_file_inventory")


## HMM Model Selection


In [ ]:
model_selection = loaded["hmm_model_selection"]
display_table(model_selection, 30)
save_table(model_selection, "study_regime_comparison_hmm_model_selection")

state_col = next((c for c in ["selected_n_states", "n_states", "states", "best_n_states"] if c in model_selection.columns), None)
if not model_selection.empty and state_col:
    label_cols = [c for c in ["asset", "frequency", "feature_set"] if c in model_selection.columns]
    labels = model_selection[label_cols].astype(str).agg(" ".join, axis=1) if label_cols else model_selection.index.astype(str)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(labels, model_selection[state_col])
    ax.set_title("Selected HMM state count by thesis-scope dataset")
    ax.set_ylabel(state_col)
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    save_fig(fig, "study_regime_comparison_hmm_selected_state_count", FIGURE_DIRS["regime_comparison"])
    plt.show()
else:
    print("HMM model-selection state column is not available.")


## HMM Posterior Confidence


In [ ]:
hmm_labels = loaded["hmm_labels"]
confidence_col = next((c for c in ["regime_confidence", "posterior_probability", "max_posterior", "confidence"] if c in hmm_labels.columns), None)
reg = regime_column(hmm_labels)
if hmm_labels.empty or not confidence_col:
    print("No HMM posterior confidence column is available.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    hmm_labels[confidence_col].dropna().plot(kind="hist", bins=40, ax=ax, color="#4C78A8")
    ax.set_title("HMM regime confidence distribution")
    ax.set_xlabel(confidence_col)
    fig.tight_layout()
    save_fig(fig, "study_regime_comparison_hmm_confidence_histogram", FIGURE_DIRS["regime_comparison"])
    plt.show()

    if reg:
        groups = [part[confidence_col].dropna().to_numpy() for _, part in hmm_labels.groupby(reg)]
        labels_order = [str(k) for k, _ in hmm_labels.groupby(reg)]
        if groups:
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.boxplot(groups, labels=labels_order, showfliers=False)
            ax.set_title("HMM confidence by regime label")
            ax.set_xlabel("Regime label")
            ax.set_ylabel(confidence_col)
            fig.tight_layout()
            save_fig(fig, "study_regime_comparison_hmm_confidence_by_regime", FIGURE_DIRS["regime_comparison"])
            plt.show()


## ARI and NMI Evaluation


In [ ]:
comparison = loaded["hmm_quantile_comparison"]
display_table(comparison, 30)
save_table(comparison, "study_regime_comparison_ari_nmi")

def plot_metric_bars(df, metric, filename):
    if df.empty or metric not in df.columns:
        print(f"{metric} is not available.")
        return
    label_cols = [c for c in ["asset", "frequency", "quantile_method", "regime_method"] if c in df.columns]
    labels = df[label_cols].astype(str).agg(" / ".join, axis=1) if label_cols else df.index.astype(str)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(labels, df[metric])
    ax.set_title(f"{metric} by asset/frequency/quantile method")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["regime_comparison"])
    plt.show()

ari_col = next((c for c in comparison.columns if c.lower() in ["adjusted_rand_index", "ari"]), None) if not comparison.empty else None
nmi_col = next((c for c in comparison.columns if c.lower() in ["normalized_mutual_information", "nmi"]), None) if not comparison.empty else None
if ari_col:
    plot_metric_bars(comparison, ari_col, "study_regime_comparison_adjusted_rand_index")
if nmi_col:
    plot_metric_bars(comparison, nmi_col, "study_regime_comparison_normalized_mutual_information")
if not ari_col and not nmi_col:
    print("ARI/NMI columns are not available in the comparison table.")


## Confusion Heatmaps


In [ ]:
confusion = loaded["hmm_quantile_confusion"]
display_table(confusion, 20)
save_table(confusion, "study_regime_comparison_confusion_table_raw")

for asset in THESIS_SCOPE_ASSETS:
    for frequency in THESIS_SCOPE_FREQUENCIES:
        filters = {"asset": asset, "frequency": frequency}
        if "quantile_method" in confusion.columns:
            methods = confusion["quantile_method"].dropna().astype(str).unique()
            preferred = "quantile_2_rolling_240"
            filters["quantile_method"] = preferred if preferred in methods else (methods[0] if len(methods) else preferred)
        plot_heatmap_from_table(
            confusion,
            f"{asset} {frequency} HMM vs quantile confusion",
            f"study_regime_comparison_confusion_{asset}_{frequency}",
            FIGURE_DIRS["regime_comparison"],
            filters,
        )


## Regime Persistence Comparison


In [ ]:
def extract_self_transition(df, method_family):
    if df is None or df.empty:
        return pd.DataFrame()
    work = df.copy()
    from_col = next((c for c in ["from_regime", "source_regime", "regime_from"] if c in work.columns), None)
    to_col = next((c for c in ["to_regime", "target_regime", "regime_to"] if c in work.columns), None)
    prob_col = next((c for c in ["self_transition_probability", "probability", "transition_probability"] if c in work.columns and pd.api.types.is_numeric_dtype(work[c])), None)
    if "self_transition_probability" in work.columns:
        out = work.copy()
        out["self_transition_probability"] = out["self_transition_probability"]
    elif from_col and to_col and prob_col:
        out = work[work[from_col].astype(str) == work[to_col].astype(str)].copy()
        out["self_transition_probability"] = out[prob_col]
        out["regime_label"] = out[from_col]
    else:
        return pd.DataFrame()
    out["method_family"] = method_family
    keep = [c for c in ["method_family", "asset", "frequency", "regime_method", "regime_label", "self_transition_probability"] if c in out.columns]
    return out[keep]

self_transitions = pd.concat([
    extract_self_transition(loaded["quantile_transitions"], "quantile"),
    extract_self_transition(loaded["hmm_persistence"], "HMM"),
], ignore_index=True)
display_table(self_transitions, 30)
save_table(self_transitions, "study_regime_comparison_self_transition_probabilities")

if not self_transitions.empty and "self_transition_probability" in self_transitions.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    labels = self_transitions[[c for c in ["method_family", "asset", "frequency", "regime_label"] if c in self_transitions.columns]].astype(str).agg(" / ".join, axis=1)
    ax.bar(labels, self_transitions["self_transition_probability"])
    ax.set_title("Self-transition probability comparison")
    ax.set_ylabel("Self-transition probability")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    save_fig(fig, "study_regime_comparison_self_transition_probabilities", FIGURE_DIRS["regime_comparison"])
    plt.show()
else:
    print("Self-transition probabilities could not be inferred from available files.")


## Key findings
Quantile and HMM comparisons should be read from the ARI, NMI, confusion, confidence, and persistence tables above when available.

## Thesis-safe interpretation
Quantile and HMM regimes do not need to match exactly. Their disagreement is informative because quantile regimes partition observed volatility directly, while HMM regimes infer latent states from multiple features. For motif discovery, both provide alternative market-state partitions under nonstationarity.

## Limitations
Empty comparison or confusion tables mean no numerical alignment claim should be made from this notebook. Missing confidence columns mean posterior-confidence figures cannot be reported.

## Recommended figures for thesis
- HMM selected state count
- HMM confidence histogram if posterior columns are available
- ARI and NMI bar charts if comparison rows are available
- HMM-vs-quantile confusion heatmaps when the confusion table is populated
